# Подготовка данных для анализа

Этот ноутбук:
- скачивает сырые события с сервера;
- объединяет несколько аккаунтов одного человека;
- подмешивает возраст и гендер;
- сохраняет готовые таблицы для следующих ноутбуков.


In [29]:
from pathlib import Path
import pandas as pd
import notebook_utils as nu

## Настройки

Если данные уже скачаны, шаг загрузки с сервера можно пропустить.

In [30]:
SERVER = "http://45.159.211.104:8000"
API_KEY = "admin1234ø"

DATA_DIR = Path("data")
RAW_PATH = DATA_DIR / "raw_events.jsonl"

## Словари соответствий


In [31]:
account_to_participant = {
    "lera": "participant_01",
    "junya": "participant_01",

    "rocty": "participant_02",
    "valya": "participant_02",
    "rocty_the_second": "participant_02",

    "piski99": "participant_03",

    "bay": "participant_04",

    "zzz": "participant_05",
    "zzzz": "participant_05",
    "zzzzzz": "participant_05",

    "sqrrl": "participant_06",

    "dashakar": "participant_07",
    "darkarma": "participant_07",

    "book": "participant_08",

    "cliawerda": "participant_09",

    "serka": "participant_10",

    "arteiy.karpov.13": "participant_11",

    "sasha": "participant_12",
    "kakashik02": "participant_12",

    "lsplsp2026.": "participant_13",

    "dunduk": "participant_14",

    "barsik_03": "participant_15",

    "egor": "participant_16",

    "hoki003": "participant_17",

    "katya_123": "participant_18",

    "rabbit": "participant_19",

    "aeplesovskikh": "participant_20",

    "ruf": "participant_21",

    "lenusya": "participant_22",

    "afl": "participant_23",

    "sweetjp": "participant_24",

    "sanchous": "participant_25",

    "alex": "participant_26",

    "lexus": "participant_27",

    "happy_elwis": "participant_28",

    "yana_n": "participant_29",

    "zubenko": "participant_30",
    "zubenko1": "participant_30",

    "ivan": "participant_31",

    "vannav": "participant_32",

    "danya": "participant_33",

    "tata": "participant_34",
}


In [32]:
participant_meta = {
    "participant_11": {"age_group": "0-17", "age_years": 12, "gender": "male", "comment": ""},
    
    "participant_13": {"age_group": "18-20", "age_years": 18, "gender": "female", "comment": ""},
    
    "participant_16": {"age_group": "18-20", "age_years": 20, "gender": "male", "comment": ""},
    "participant_17": {"age_group": "18-20", "age_years": 20, "gender": "male", "comment": ""},
    
    "participant_14": {"age_group": "21-24", "age_years": 22, "gender": "male", "comment": ""},
    "participant_15": {"age_group": "21-24", "age_years": 22, "gender": "male", "comment": ""},
    
    "participant_27": {"age_group": "21-24", "age_years": 23, "gender": "male", "comment": ""},
    
    "participant_20": {"age_group": "21-24", "age_years": 24, "gender": "male", "comment": ""},
    "participant_33": {"age_group": "21-24", "age_years": 24, "gender": "male", "comment": ""},

    "participant_01": {"age_group": "25-28", "age_years": 25, "gender": "female", "comment": ""},
    "participant_03": {"age_group": "25-28", "age_years": 25, "gender": "female", "comment": ""},
    "participant_05": {"age_group": "25-28", "age_years": 25, "gender": "female", "comment": ""},
    "participant_06": {"age_group": "25-28", "age_years": 25, "gender": "male", "comment": ""}, 
    "participant_09": {"age_group": "25-28", "age_years": 25, "gender": "female", "comment": ""},
    "participant_12": {"age_group": "25-28", "age_years": 25, "gender": "male", "comment": ""},
    "participant_18": {"age_group": "25-28", "age_years": 25, "gender": "female", "comment": ""},
    
    "participant_04": {"age_group": "25-28", "age_years": 26, "gender": "male", "comment": ""},
    "participant_19": {"age_group": "25-28", "age_years": 26, "gender": "male", "comment": ""},
    "participant_32": {"age_group": "25-28", "age_years": 26, "gender": "female", "comment": ""},
    
    "participant_07": {"age_group": "25-28", "age_years": 28, "gender": "female", "comment": ""},
    "participant_29": {"age_group": "25-28", "age_years": 28, "gender": "female", "comment": ""},
    
    "participant_02": {"age_group": "29-34", "age_years": 30, "gender": "male", "comment": ""},
    "participant_21": {"age_group": "29-34", "age_years": 30, "gender": "male", "comment": ""},
    "participant_22": {"age_group": "29-34", "age_years": 30, "gender": "female", "comment": ""},
    "participant_23": {"age_group": "29-34", "age_years": 30, "gender": "male", "comment": ""},
    "participant_24": {"age_group": "29-34", "age_years": 30, "gender": "male", "comment": ""},
    "participant_26": {"age_group": "29-34", "age_years": 30, "gender": "male", "comment": ""},
    
    "participant_08": {"age_group": "29-34", "age_years": 31, "gender": "male", "comment": ""},
    "participant_25": {"age_group": "29-34", "age_years": 31, "gender": "female", "comment": ""},
    "participant_28": {"age_group": "29-34", "age_years": 31, "gender": "male", "comment": ""},
    
    "participant_34": {"age_group": "29-34", "age_years": 34, "gender": "female", "comment": ""},
    
    "participant_30": {"age_group": "35-39", "age_years": 35, "gender": "male", "comment": ""},
    "participant_31": {"age_group": "35-39", "age_years": 35, "gender": "male", "comment": ""},
    
    "participant_10": {"age_group": "35-39", "age_years": 39, "gender": "male", "comment": ""},
}

## Скачать сырые события с сервера

In [33]:
raw_rows = nu.fetch_raw_events(server=SERVER, api_key=API_KEY)
nu.save_jsonl(RAW_PATH, raw_rows)
print(f"Downloaded rows: {len(raw_rows)}")

Downloaded rows: 15871


## Загрузить raw events из локального файла

In [34]:
raw_rows = nu.load_jsonl(RAW_PATH)
print(f"Loaded raw rows: {len(raw_rows)}")

Loaded raw rows: 15871


## Собрать таблицы

Здесь формируются:
- `task_table` — одна строка на одну задачу;
- `adaptation_table` — одна строка на один шаг адаптации;
- `session_table` — одна строка на одну сессию;
- `participant_mode_table` — агрегирование по участнику и режиму.

## Что означают признаки в таблицах

`task_table`:
- `event_id` — уникальный идентификатор события в сырых данных;
- `event_ts` — исходная временная метка события в строковом формате;
- `event_datetime` — та же временная метка, преобразованная в удобный формат даты и времени;
- `session_id` — идентификатор игровой сессии;
- `account_id` — аккаунт, под которым играл пользователь;
- `participant_id` — общий идентификатор участника после объединения нескольких аккаунтов;
- `mode` — режим адаптации (`baseline` или `ppo`);
- `task_id` — тип мини-игры;
- `batch_index` — номер батча внутри сессии;
- `batch_task_index` — номер задачи внутри батча;
- `level` — уровень сложности на момент задачи;
- `reaction_time` — время ответа в миллисекундах;
- `correct` — правильно ли решена задача;
- `deadline_met` — успел ли пользователь ответить до конца временного окна;
- `answered` — бинарный признак, был ли вообще дан ответ;
- `solved` — бинарный признак, была ли задача успешно решена;
- `question_text` — текст вопроса, если он был сохранён в payload;
- `rule` — активное правило в задаче на переключение правил;
- `target_symbol` — целевой символ в задаче radar scan;
- `age_group`, `gender` — внешние признаки участника.
- `age_years` — возраст участника в годах, если он был задан во внешнем словаре;
- `comment` — дополнительная произвольная пометка по участнику.

`adaptation_table`:
- `event_id` — уникальный идентификатор события адаптации;
- `event_ts` — исходная временная метка события;
- `event_datetime` — временная метка в формате даты и времени;
- `session_id` — идентификатор сессии, в которой произошёл шаг адаптации;
- `account_id` — аккаунт пользователя;
- `participant_id` — объединённый идентификатор участника;
- `mode` — режим адаптации;
- `step` — номер шага адаптации;
- `batch_index` — номер батча, после которого выполнялся шаг адаптации;
- `batch_tasks_completed` — сколько задач было завершено к этому моменту;
- `action_id` — идентификатор выбранного действия в пространстве действий модели;
- `delta_level` — изменение уровня, если оно было предусмотрено в записи;
- `delta_tempo` — как модель изменила темп (`-1`, `0`, `+1`);
- `tempo_offset` — текущий накопленный сдвиг темпа;
- `reward` — награда адаптационного шага;
- `level` — уровень сложности, на котором был сделан шаг адаптации;
- `action_space` — обозначение пространства действий модели;
- `state_accuracy`, `state_mean_rt` — состояние игрока перед действием модели;
- `state_std_rt` — разброс времени ответа в текущем окне наблюдения;
- `state_error_streak` — длина серии ошибок перед действием модели;
- `state_switch_cost` — оценка стоимости переключения в задачах на rule switch;
- `state_fatigue_trend` — оценка утомления по изменению времени ответа;
- `state_level` — уровень, зафиксированный внутри вектора состояния;
- `task_offset_*` — персональные смещения сложности по конкретным мини-играм.
- `age_group`, `gender`, `age_years`, `comment` — внешние признаки участника.

`session_table`:
- `participant_id` — идентификатор участника;
- `account_id` — аккаунт, под которым игралась сессия;
- `session_id` — идентификатор сессии;
- `mode` — режим адаптации в данной сессии;
- `age_group`, `gender` — внешние признаки участника;
- `total_tasks` — сколько задач было в сессии;
- `answered_tasks` — сколько задач пользователь успел ответить;
- `solved_tasks` — сколько задач было решено правильно;
- `accuracy_total` — доля правильных решений среди всех задач;
- `accuracy_answered` — доля правильных решений только среди отвеченных задач;
- `answered_rate` — доля задач, на которые успели ответить;
- `mean_rt_ms`, `median_rt_ms`, `p90_rt_ms` — характеристики времени ответа;
- `first_level` — уровень в начале сессии;
- `last_level` — уровень в конце сессии;
- `max_level` — максимальный достигнутый уровень в сессии;
- `level_gain` — разница между последним и первым уровнем;
- `completed_batches` — сколько батчей задач было фактически завершено;
- `task_span_minutes` — длительность сессии в минутах между первой и последней задачей.

`participant_mode_table`:
- одна строка соответствует одному участнику в одном режиме;
- `participant_id` — идентификатор участника;
- `mode` — режим адаптации;
- `age_group`, `gender` — внешние признаки участника;
- `sessions` — сколько сессий вошло в агрегирование;
- `total_tasks` — суммарное число задач по всем сессиям участника в данном режиме;
- `answered_tasks` — суммарное число задач, на которые участник ответил;
- `solved_tasks` — суммарное число задач, решённых правильно;
- `accuracy_total` — общая доля правильных решений среди всех задач;
- `accuracy_answered` — доля правильных решений среди отвеченных задач;
- `answered_rate` — общая доля задач с ответом;
- `mean_rt_ms` — среднее время ответа по сессиям участника;
- `median_rt_ms` — средняя медиана времени ответа по сессиям участника;
- `p90_rt_ms` — среднее значение верхнего квантиля времени ответа по сессиям;
- `max_level` — максимальный достигнутый уровень в данном режиме;
- `level_gain_mean` — средний прирост уровня за сессию;
- `solved_tasks_per_session` — среднее число правильно решённых задач на сессию;
- `answered_tasks_per_session` — среднее число задач с ответом на сессию;
- `total_tasks_per_session` — среднее число задач на сессию;
- остальные метрики показывают усреднённое качество прохождения в данном режиме.

In [35]:
task_df = nu.rows_to_task_table(raw_rows)
adaptation_df = nu.rows_to_adaptation_table(raw_rows)

task_df = nu.attach_participant_columns_from_dicts(task_df, account_to_participant=account_to_participant, participant_meta=participant_meta)
adaptation_df = nu.attach_participant_columns_from_dicts(adaptation_df, account_to_participant=account_to_participant, participant_meta=participant_meta)

# до этого дня считаем, что модель была не обучена, поэтому считаем сесси в модельном режиме как сессии в базовом режиме
cutoff_date = pd.Timestamp("2026-04-06", tz="UTC")
task_df.loc[(task_df["mode"] == "ppo") & (task_df["event_datetime"] < cutoff_date), "mode"] = "baseline"
adaptation_df.loc[(adaptation_df["mode"] == "ppo") & (adaptation_df["event_datetime"] < cutoff_date),"mode"] = "baseline"

In [36]:
session_df = nu.build_session_table(task_df)
participant_mode_df = nu.build_participant_mode_table(session_df)

In [37]:
print("task_table:", task_df.shape)
print("adaptation_table:", adaptation_df.shape)
print("session_table:", session_df.shape)
print("participant_mode_table:", participant_mode_df.shape)

task_table: (14158, 23)
adaptation_table: (1393, 33)
session_table: (82, 21)
participant_mode_table: (44, 19)


In [38]:
# посмотрим кто уже поиграл в обновленную версию
participant_mode_df[participant_mode_df['mode']=='ppo']

,participant_id,mode,age_group,gender,sessions,total_tasks,answered_tasks,solved_tasks,accuracy_total,accuracy_answered,answered_rate,mean_rt_ms,median_rt_ms,p90_rt_ms,max_level,level_gain_mean,solved_tasks_per_session,answered_tasks_per_session,total_tasks_per_session
0,biba,ppo,unknown,unknown,1,420,418,377,0.897619,0.901914,0.995238,1572.789474,1281.500,2981.900,10.0,9.0,377.00,418.00,420.0
3,participant_01,ppo,25-28,female,1,30,9,8,0.266667,0.888889,0.300000,1991.888889,1853.000,3341.200,10.0,0.0,8.00,9.00,30.0
6,participant_03,ppo,25-28,female,1,234,229,206,0.880342,0.899563,0.978632,1822.820961,1563.000,3150.800,10.0,9.0,206.00,229.00,234.0
9,participant_05,ppo,25-28,female,1,162,159,135,0.833333,0.849057,0.981481,1760.962264,1470.000,3092.800,10.0,9.0,135.00,159.00,162.0
12,participant_07,ppo,25-28,female,1,150,147,139,0.926667,0.945578,0.980000,1829.265306,1562.000,3083.000,10.0,9.0,139.00,147.00,150.0
14,participant_08,ppo,29-34,male,2,168,167,147,0.875000,0.880240,0.994048,1615.393972,1316.000,2723.900,10.0,4.0,73.50,83.50,84.0
17,participant_10,ppo,35-39,male,2,60,60,55,0.916667,0.916667,1.000000,1931.833333,1686.500,2939.800,3.0,1.5,27.50,30.00,30.0
38,participant_30,ppo,35-39,male,4,120,119,117,0.975000,0.983193,0.991667,1697.425000,1567.375,2603.175,9.0,2.0,29.25,29.75,30.0


## Сохранить подготовленные таблицы

In [39]:
nu.save_prepared_tables(out_dir=DATA_DIR, task_df=task_df, session_df=session_df, participant_mode_df=participant_mode_df, adaptation_df=adaptation_df)
print(f"Saved to: {DATA_DIR}")

Saved to: data
